# Quiz 7 — Trees: House Listings
## 完整答案

**变量：** bedrooms, bathrooms, area, lot_size, floors, year_built→age, garage, fireplace, renovated, neighborhood, distance_city → 预测 **price**

---
### 答案速查表

| Q | 答案 |
|---|------|
| Q1  | Train avg age = **50.13** |
| Q2  | 与 price 最强相关 = **area** (r ≈ 0.933) |
| Q3  | age=3, depth=2 → **813,010** |
| Q4  | age=3, garage=1, depth=2 → **813,010** |
| Q5  | age=3, garage=1, depth=4 → **822,033** |
| Q6  | Train RMSE (age+garage, depth=2) = **213,550** |
| Q7  | Train RMSE (5 feats, depth=2) = **116,437** |
| Q8  | Train RMSE (all, depth=2) = **99,343** |
| Q9  | Test RMSE (all, depth=2) = **99,331** |
| Q10 | Train RMSE (all, depth=10) = **46,516** |
| Q11 | Test RMSE (all, depth=10) = **59,709** |
| Q12 | Train RMSE (maximal) = **0** |
| Q13 | Test RMSE (maximal) = **75,173** |
| Q14 | 最优 max_depth = **8** |
| Q15 | 最优参数 = **max_depth=9, min_samples_leaf=20, max_features=12** |
| Q16 | Tuned Train RMSE = **51,488** |
| Q17 | Tuned Test RMSE = **57,694** |

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.tree import DecisionTreeRegressor, plot_tree
from sklearn.metrics import mean_squared_error
import matplotlib.pyplot as plt

def rmse(y_true, y_pred):
    return round(np.sqrt(mean_squared_error(y_true, y_pred)), 2)

---
### Q1
**原题：** Read data. Create `age = 2023 - year_built`. Stratified split 70/30 (strata = pd.qcut(price, q=100)), random_state=617. What is the average age in train?

In [ ]:
data = pd.read_csv('house_listings.csv')
data['age'] = 2023 - data['year_built']

y = data['price']
X = data.drop(columns=['price', 'year_built'])   # 删除 year_built，保留 age

# 分层变量：price 分 100 个等频 bin
y_binned = pd.qcut(data['price'], q=100)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, train_size=0.7, random_state=617, stratify=y_binned)

print(f'Train shape: {X_train.shape}, Test shape: {X_test.shape}')
print(f'Avg age (train): {round(X_train["age"].mean(), 2)}')
# ✅ 答案：50.13

---
### Q2
**原题：** Which feature is most strongly correlated with price?

In [ ]:
train_full = X_train.copy()
train_full['price'] = y_train.values

corr_price = train_full.corr()['price'].drop('price').abs().sort_values(ascending=False)
print('与 price 的相关系数（绝对值，降序）：')
print(corr_price.round(4))
# ✅ 答案：area (r ≈ 0.933)

---
### Q3
**原题：** Fit regression tree with `age`, max_depth=2. Predicted price for age=3?

In [ ]:
tree3 = DecisionTreeRegressor(max_depth=2, random_state=617)
tree3.fit(X_train[['age']], y_train)

# 可视化树
plt.figure(figsize=(10, 4))
plot_tree(tree3, feature_names=['age'], filled=True, rounded=True, fontsize=10)
plt.title('Regression Tree: age only, max_depth=2')
plt.show()

pred3 = tree3.predict(pd.DataFrame({'age': [3]}))
print(f'Predicted price (age=3): {round(pred3[0], 2)}')
# ✅ 答案：813,010（选项中最接近的是 813010）

---
### Q4
**原题：** Fit regression tree with `age` + `garage`, max_depth=2. Predicted price for age=3, garage=1?

In [ ]:
tree4 = DecisionTreeRegressor(max_depth=2, random_state=617)
tree4.fit(X_train[['age', 'garage']], y_train)

plt.figure(figsize=(12, 4))
plot_tree(tree4, feature_names=['age','garage'], filled=True, rounded=True, fontsize=9)
plt.title('Regression Tree: age+garage, max_depth=2')
plt.show()

pred4 = tree4.predict(pd.DataFrame({'age': [3], 'garage': [1]}))
print(f'Predicted price (age=3, garage=1): {round(pred4[0], 2)}')
# ✅ 答案：813,010
# 📌 注意：garage 在 depth=2 时未被选为分裂节点（age 已解释足够多），所以结果与 Q3 相同

---
### Q5
**原题：** Same as Q4 but max_depth=4. Predicted price for age=3, garage=1?

In [ ]:
tree5 = DecisionTreeRegressor(max_depth=4, random_state=617)
tree5.fit(X_train[['age', 'garage']], y_train)

plt.figure(figsize=(18, 6))
plot_tree(tree5, feature_names=['age','garage'], filled=True, rounded=True, fontsize=8)
plt.title('Regression Tree: age+garage, max_depth=4')
plt.show()

pred5 = tree5.predict(pd.DataFrame({'age': [3], 'garage': [1]}))
print(f'Predicted price (age=3, garage=1, depth=4): {round(pred5[0], 2)}')
# ✅ 答案：822,033

---
### Q6
**原题：** Regression tree with age + garage, max_depth=2. Train RMSE?

In [ ]:
# 复用 tree4
train_rmse_q6 = rmse(y_train, tree4.predict(X_train[['age','garage']]))
print(f'Train RMSE (age+garage, depth=2): {train_rmse_q6}')
# ✅ 答案：213,550

---
### Q7
**原题：** Regression tree with age, garage, bedrooms, bathrooms, neighborhood. max_depth=2. Train RMSE?

In [ ]:
feats7 = ['age', 'garage', 'bedrooms', 'bathrooms', 'neighborhood']
tree7 = DecisionTreeRegressor(max_depth=2, random_state=617)
tree7.fit(X_train[feats7], y_train)
print(f'Train RMSE (5 feats, depth=2): {rmse(y_train, tree7.predict(X_train[feats7]))}')
# ✅ 答案：116,437

---
### Q8
**原题：** All predictors, max_depth=2. Train RMSE?

In [ ]:
tree8 = DecisionTreeRegressor(max_depth=2, random_state=617)
tree8.fit(X_train, y_train)
print(f'Train RMSE (all feats, depth=2): {rmse(y_train, tree8.predict(X_train))}')
# ✅ 答案：99,343

---
### Q9
**原题：** Same model as Q8. Test RMSE?

In [ ]:
print(f'Test RMSE (all feats, depth=2): {rmse(y_test, tree8.predict(X_test))}')
# ✅ 答案：99,331
# 📌 Train ≈ Test → 没有过拟合，但模型太简单（depth=2 限制了表达能力）

---
### Q10
**原题：** All predictors, max_depth=10. Train RMSE?

In [ ]:
tree10 = DecisionTreeRegressor(max_depth=10, random_state=617)
tree10.fit(X_train, y_train)
print(f'Train RMSE (all, depth=10): {rmse(y_train, tree10.predict(X_train))}')
# ✅ 答案：46,516

---
### Q11
**原题：** Same model as Q10. Test RMSE?

In [ ]:
print(f'Test RMSE (all, depth=10): {rmse(y_test, tree10.predict(X_test))}')
# ✅ 答案：59,709
# 📌 Train(46,516) < Test(59,709) → 开始出现过拟合（depth越深，过拟合越严重）

---
### Q12
**原题：** Maximal tree (no constraints). Train RMSE?

In [ ]:
tree12 = DecisionTreeRegressor(random_state=617)  # 不设任何约束
tree12.fit(X_train, y_train)
print(f'Train RMSE (maximal): {rmse(y_train, tree12.predict(X_train))}')
print(f'Max depth of tree: {tree12.get_depth()}')
# ✅ 答案：0.0（完全记忆训练数据 → 极端过拟合）

---
### Q13
**原题：** Maximal tree. Test RMSE?

In [ ]:
print(f'Test RMSE (maximal): {rmse(y_test, tree12.predict(X_test))}')
# ✅ 答案：75,173
# 📌 Train=0 但 Test=75,173 → 教科书级别的过拟合案例
# 📌 RMSE 对比：
#    depth=2:   Train=99k,  Test=99k  → 欠拟合（bias高）
#    depth=10:  Train=47k,  Test=60k  → 轻微过拟合
#    maximal:   Train=0,    Test=75k  → 严重过拟合（variance高）

---
### Q14
**原题：** Grid search on max_depth=[2,3,5,8,12], cv=5, scoring=RMSE. Best max_depth?

In [ ]:
param_grid_14 = {'max_depth': [2, 3, 5, 8, 12]}
gs14 = GridSearchCV(
    DecisionTreeRegressor(random_state=617),
    param_grid_14,
    cv=5,
    scoring='neg_root_mean_squared_error')
gs14.fit(X_train, y_train)

print(f'Best max_depth: {gs14.best_params_["max_depth"]}')
print('\n所有结果：')
for params, score in zip(gs14.cv_results_['params'], gs14.cv_results_['mean_test_score']):
    print(f'  depth={params["max_depth"]}: CV RMSE = {round(-score, 2)}')
# ✅ 答案：max_depth = 8 (CV RMSE ≈ 58,430)

---
### Q15
**原题：** Grid search on max_depth, min_samples_leaf, max_features. cv=5. Best model?

In [ ]:
param_grid_15 = {
    'max_depth':        [3, 8, 9, 12],
    'min_samples_leaf': [5, 20, 100],
    'max_features':     [2, 4, 12]
}
gs15 = GridSearchCV(
    DecisionTreeRegressor(random_state=617),
    param_grid_15,
    cv=5,
    scoring='neg_root_mean_squared_error')
gs15.fit(X_train, y_train)

print(f'Best params: {gs15.best_params_}')
print(f'Best CV RMSE: {round(-gs15.best_score_, 2)}')
# ✅ 答案：max_depth=9, min_samples_leaf=20, max_features=12
# 📌 max_features 控制每次分裂时随机考虑的特征数量（类似 Random Forest 的机制）

---
### Q16
**原题：** Tuned model train RMSE?

In [ ]:
best_tree = gs15.best_estimator_
print(f'Tuned Train RMSE: {rmse(y_train, best_tree.predict(X_train))}')
# ✅ 答案：51,488

---
### Q17
**原题：** Tuned model test RMSE?

In [ ]:
print(f'Tuned Test RMSE: {rmse(y_test, best_tree.predict(X_test))}')
# ✅ 答案：57,694
# 📌 最终模型对比（Test RMSE）：
#    depth=2 (简单):     99,331  → 欠拟合
#    depth=10:           59,709  → 轻微过拟合
#    maximal (无约束):   75,173  → 严重过拟合
#    tuned (GridSearch): 57,694  → 最优！

---
## 核心考点总结

| 概念 | 要点 |
|------|------|
| **Bias-Variance Tradeoff** | 树越深 → variance↑（过拟合），树越浅 → bias↑（欠拟合）|
| **过拟合信号** | Train RMSE << Test RMSE |
| **欠拟合信号** | Train RMSE ≈ Test RMSE 但两者都很大 |
| **Maximal Tree** | Train RMSE = 0，完全记忆训练数据 |
| **GridSearchCV** | scoring='neg_root_mean_squared_error'，越大（越接近0）越好 |
| **pd.qcut(q=100)** | 等频分箱用于分层抽样，保证 train/test price 分布一致 |
| **max_features** | 每次分裂随机选取的特征数，减少相关性，防止过拟合 |
| **min_samples_leaf** | 叶节点最少样本数，越大 → 树越简单 |